In [ ]:
# Importing libraries 

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import pingouin as pg
from statsmodels.stats.multitest import fdrcorrection
%matplotlib inline

In [ ]:
# Importing cluster Volume data (2 sample T-Test, HO vs MCI, corrected, p<0.05) 

GM_cluster_vol_df = pd.read_csv('data/final_df.csv', 
                                sep = ',', encoding = 'utf-8', low_memory = False)
GM_cluster_vol_df = GM_cluster_vol_df.rename(columns={'Subjects': 'names'})
GM_cluster_vol_df.head()

In [ ]:
GM_cluster_vol_df.shape

In [ ]:
# Remove duplicate rows based on 'names' column
GM_cluster_vol_df = GM_cluster_vol_df.drop_duplicates(subset=['CODE'])
GM_cluster_vol_df.shape

In [ ]:
GM_cluster_vol_df = GM_cluster_vol_df.rename(columns={'CODE': 'names'})

In [ ]:
# Importing Cognitive test data :

Cogn_df = pd.read_csv('data/All_cognitive_scores_MCI_baseline.csv',
                      sep = ',', encoding = 'utf-8', low_memory = False)
Cogn_df.head()

In [ ]:
Cogn_df.shape

In [ ]:
# Merge data of interest based on subject name column :
merged_df = pd.merge(Cogn_df, GM_cluster_vol_df, on='names')
merged_df.head()

In [ ]:
# Combine left and right hippocampus into bilateral hippocampus
merged_df['B_Hippocampus'] = merged_df['L_Hippocampus'] + merged_df['R_Hippocampus']
merged_df[['names', 'L_Hippocampus', 'R_Hippocampus', 'B_Hippocampus']].head()

In [ ]:
# QQ plot and histogram for B_Hippocampus
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Histogram
axes[0].hist(merged_df['B_Hippocampus'].dropna(), bins=15, edgecolor='black')
axes[0].set_title('Histogram of B_Hippocampus')
axes[0].set_xlabel('B_Hippocampus')
axes[0].set_ylabel('Frequency')

# QQ plot
stats.probplot(merged_df['B_Hippocampus'].dropna(), dist="norm", plot=axes[1])
axes[1].set_title('QQ Plot of B_Hippocampus')

plt.tight_layout()
plt.show()

# Shapiro-Wilk test
stat, p = stats.shapiro(merged_df['B_Hippocampus'].dropna())
print(f"Shapiro-Wilk test: W={stat:.4f}, p={p:.5f}")
if p > 0.05:
    print("=> Normal distribution (fail to reject H0)")
else:
    print("=> Non-normal distribution (reject H0)")

In [ ]:
merged_df.columns.to_list()

In [ ]:
# Load the correlation type specifications
corr_spec = pd.read_csv('data/Correlation_tests.csv', sep=',', encoding='utf-8', low_memory=False)
corr_spec.set_index('Test / Cluster', inplace=True)

# Get rid of blanks in columns and index
corr_spec.index = corr_spec.index.str.strip()
corr_spec.columns = corr_spec.columns.str.strip()

In [ ]:
# Iterate through each test and cluster pair
for test in corr_spec.index:
    for cluster in corr_spec.columns:
        corr_type = corr_spec.loc[test, cluster]
        print(f"Test: {test}, Cluster: {cluster}, Correlation Type: {corr_type}")

In [ ]:
# Pearson with CI
result_pearson = pg.corr(merged_df["L_Amygdala"], merged_df["MMSE"], method='pearson')
print(result_pearson[['n', 'r', 'CI95%', 'p-val']])

# Kendall with CI
# result_kendall = pg.corr(merged_df["R_Hippocampus"], merged_df["MMSE"], method='kendall')
# print(result_kendall[['n', 'r', 'CI95%', 'p-val']])

In [ ]:
# Using a list to store multiple result dictionaries
results = []

# OR using a dictionary with lists for each field
results_dict = {
    'Test': [],
    'Cluster': [],
    'N': [],
    'Correlation_Type': [],
    'Correlation': [],
    'P_value': [],
    'CI95%': []
}

In [ ]:
# Iterate through each test and cluster pair
for test in corr_spec.index:
    for cluster in corr_spec.columns:
        corr_type = corr_spec.loc[test, cluster]
        try:
            # Create a temporary dataframe with both columns
            temp_df = merged_df[[test, cluster]].dropna()
            test_clean = temp_df[test]
            cluster_clean = temp_df[cluster]

            #print(f"Sample size: {len(test_clean)}")

            if corr_type == 'Pearson':
                # Calculate Pearson correlation
                # corr, p_value = stats.pearsonr(test_clean, cluster_clean)
                result_corr = pg.corr(test_clean, cluster_clean, method='pearson')

            elif corr_type == 'Kendall':
                # Calculate Kendall correlation
                # corr, p_value = stats.kendalltau(test_clean, cluster_clean)
                result_corr = pg.corr(test_clean, cluster_clean, method='kendall')

            # For each pair you calculate manually, add like this:
            results.append({
                'Test': test,
                'Cluster': cluster,
                'N': result_corr['n'][0],
                'Correlation_Type': corr_type,
                'Correlation': round(result_corr['r'][0], 3),
                'P_value': round(result_corr['p-val'][0], 5),
                'CI95%': list(result_corr['CI95%'][0])
            })
        except KeyError:
            # Continue to next iteration if test or cluster not found in merged_df
            continue 

In [ ]:
# Calculate Pearson correlation
corr, p_value = stats.pearsonr(test_clean, cluster_clean)

In [ ]:
# Calculate Kendall correlation
corr, p_value = stats.kendalltau(test_clean, cluster_clean)

In [ ]:
# For each pair you calculate manually, add like this:
results.append({
    'Test': 'MMSE',
    'Cluster': 'L_Hippocampus',
    'Correlation_Type': 'Pearson',
    'Correlation': round(corr, 3),
    'P_value': round(p_value, 5)
})

In [ ]:
results

In [ ]:
# Convert results list to DataFrame
results_df = pd.DataFrame(results)

# Apply FDR correction using fdrcorrection
p_values = results_df['P_value'].values
reject, p_adjusted = fdrcorrection(p_values, alpha=0.05, method='indep')

# Add corrected p-values to dataframe
results_df['P_value_FDR'] = p_adjusted
results_df['Significant_FDR'] = reject

# Display results
results_df

In [ ]:
results_df.shape

In [ ]:
# Exporting results to CSV file
results_df.to_csv('data/Correlation_results_all_FDR_corrected_LR_HPC.csv', 
                  index=False)